In [ ]:
# Setup
from pathlib import Path
import sys, os, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from model import build_or_load_store, retrieve
from data import get_llm_answer

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
with open(ARTEFACTS_DIR / 'rag_results.json') as f:
    results = json.load(f)
print("Saved results:")
for k, v in results.items():
    print(f"  {k}: {v}")

In [ ]:
# 1. End-to-end demo on a fresh query
store = build_or_load_store(force_rebuild=False)
print(f"Store contains {len(store)} chunks\n")

demo_query = "What is the default chunk size used by the RAG bot?"
print(f"Demo query: {demo_query}\n")
hits = retrieve(demo_query, store, top_k=3)
chunks = [h['chunk'] for h in hits]

result = get_llm_answer(demo_query, chunks, backend='auto')
print("=== ANSWER ===")
print(result['answer'])
print(f"\nBackend: {result['backend']}")
print(f"\n=== SOURCES (top-3) ===")
for i, h in enumerate(hits):
    print(f"[{i+1}] {h['source']}  score={h['score']:.3f}")
    print(f"    {h['chunk'][:150]}...")

In [ ]:
# 2. Business-readable summary
lines = [
    "=== BUSINESS SUMMARY ===",
    "Business question: can we build a chatbot that answers questions from a",
    "document corpus, with citations, so employees get accurate answers in",
    "seconds instead of searching for hours?",
    "",
    f"Pipeline: PDF -> pypdf -> RecursiveCharacterTextSplitter -> sentence-transformers ->",
    f"          SimpleVectorStore (numpy + cosine) -> top-k retrieval -> LLM (OpenAI/Ollama)",
    "",
    f"Corpus size: {len(store)} chunks",
    f"LLM backend used: {result['backend']}",
    "",
    "TCD module mapping:",
    "- Generative AI for Business: RAG architecture, vector stores, embeddings,",
    "  prompt engineering, LLM evaluation, hallucination mitigation.",
    "- Business AI Deployment: dockerise this app to demonstrate MLOps / productionisation.",
    "",
    "UCD module mapping:",
    "- Managing Data & AI: data governance, AI ethics, MLOps management.",
    "",
    "Recommended next steps:",
    "- Replace the placeholder document with 10-20 public annual reports.",
    "- Dockerise the app to demonstrate Business AI Deployment module.",
    "- Add retrieval precision@k evaluation on a labelled Q&A dataset.",
]
print("\n".join(lines))